# Plot Condensibility score

##### Project Description:

This is the very first batch of plot. I'm developing base on Sangwoo's data.

##### Working Directory:

/Volumes/BackupHaLab/condense-seq/ipython_notebooks/[Fig.1d]Condensibility_score.2026.ipynb

##### Start Date:

Jan 9, 2026

##### Editor:

Xin Lin

Here is the link to [Sangwoo's paper](https://www.nature.com/articles/s41586-025-08971-7)

In [8]:
# python modules
import sys, re, os
from collections import defaultdict
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import scipy
from scipy import stats

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Matplotlib:", mpl.__version__)
print("SciPy:", scipy.__version__)

# custom modules - Customized to condense-seq/
working_dir = '/lab-share/PCMM-Ha-e2/Public/Xin'
sys.path.append(f'{working_dir}/condense-seq/postpro_scripts')
import graphics_edit as graphics
import load_file_edit as load_file
import Interval_dict_python3
import statis_edit as statis

Python: 3.9.19 | packaged by conda-forge | (main, Mar 20 2024, 12:50:21) 
[GCC 12.3.0]
NumPy: 1.26.4
Matplotlib: 3.8.4
SciPy: 1.13.1


In [9]:
# matplotlib setting
%matplotlib inline
mpl.rcParams["figure.facecolor"] = "white"
mpl.rcParams["axes.facecolor"] = "white"
mpl.rcParams["savefig.facecolor"] = "white"


In [10]:
### parameters
cell_org = {'Exp1':'mouse',
            'Exp2':'mouse',
            'Exp3':'mouse'}

cell_chrnames = {'Exp1':['chr%s' % (i) for i in range(1, 20)] + ['chrX'],
                 'Exp2':['chr%s' % (i) for i in range(1, 20)] + ['chrX'],
                 'Exp3':['chr%s' % (i) for i in range(1, 20)] + ['chrX']}
# # DEBUG
# print(cell_chrnames)

In [11]:
### chromosome choices
chr_choices = cell_chrnames['Exp2']
# chr_choices = ['chr1', 'chr2', 'chr3']

## Loading files

Here are many files loaded into the working space

In [12]:
### load gtab file
working_dir = "/Volumes/BackupHaLab/"
gtab_path = working_dir + "Oct2025_run/analysis_v2/"

dinfo_dkey = {
    # 'Exp1-Samp11_S2_score.gtab.gz': {'Exp1-Samp11_S2': (1, 'Exp1', 'score', 1)},
    # 'Exp1-Samp15_S3_score.gtab.gz': {'Exp1-Samp15_S3': (2, 'Exp1', 'score', 1)},
    # 'Exp2-Samp11_S5_score.gtab.gz': {'Exp2-Samp11_S5': (1, 'Exp2', 'score', 1)},
    # 'Exp2-Samp15_S6_score.gtab.gz': {'Exp2-Samp15_S6': (2, 'Exp2', 'score', 1)},
    # 'Exp3-Samp11_S8_score.gtab.gz': {'Exp3-Samp11_S8': (1, 'Exp3', 'score', 1)},
    # 'Exp3-Samp15_S9_score.gtab.gz': {'Exp3-Samp15_S9': (2, 'Exp3', 'score', 1)},
    # 'Exp4-Samp11_S11_score.gtab.gz': {'Exp4-Samp11_S11': (1, 'Exp4', 'score', 1)},
    # 'Exp4-Samp15_S12_score.gtab.gz': {'Exp4-Samp15_S12': (2, 'Exp4', 'score', 1)},
    # 'Exp5-Samp11_S14_score.gtab.gz': {'Exp5-Samp11_S14': (1, 'Exp5', 'score', 1)},
    # 'Exp5-Samp15_S15_score.gtab.gz': {'Exp5-Samp15_S15': (2, 'Exp5', 'score', 1)},
    'Exp6-Samp11_S17_score.gtab.gz': {'Exp6-Samp11_S17': (1, 'Exp6', 'score', 1)},
    'Exp6-Samp15_S18_score.gtab.gz': {'Exp6-Samp15_S18': (2, 'Exp6', 'score', 1)},
    # 'E14_NCP_titr15_1rep_score.gtab.gz': {'E14_NCP_titr15_1rep': (2, 'E14_NCP', 'score', 1)},
    # 'E14_NCP_titr15_2rep_score.gtab.gz': {'E14_NCP_titr15_2rep': (2, 'E14_NCP', 'score', 2)},
    # 'E14_NCP_titr11_1rep_score.gtab.gz': {'E14_NCP_titr11_1rep': (1, 'E14_NCP', 'score', 1)},
    # 'E14_NCP_titr11_2rep_score.gtab.gz': {'E14_NCP_titr11_2rep': (1, 'E14_NCP', 'score', 2)},
}

# The following function is NOWHERE to be found...
chr_dkey_ID_value = load_file.read_gtab_batch(dinfo_dkey,
                                               data_path=gtab_path,
                                               chr_choices=chr_choices,
                                               by_chr_first=True,
                                               verbal=True)
# print(list(chr_dkey_ID_value.items())[:5])

Done


In [13]:
### read titration file
titr_path = working_dir + 'Oct2025_run/analysis_v2/titration_files/'
titr_fname = 'Exp6-titration.tsv'
tnum_conc, tnum_frac = load_file.read_titration (titr_path + titr_fname)
print(tnum_conc, tnum_frac)

FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/BackupHaLab/Oct2025_run/analysis_v2/titration_files/Exp6-titration.tsv'

In [ ]:
### figure parameters
# set figure binning parameters
i = 20
bin_size =  1000 # binsize (unit of bp) - default: int(0.5*(10**6) / i)
bin_step = bin_size # no overlap
blur_win = int(4*i + 1) # sliding window (unit of bin)

In [12]:
### binning/smoothing the condense-seq data
chr_dkey_sig = {}
output_bdg = True
output_bigwig = True

def _find_inner_key_by_value(outer_dict, value):
    for outer_key, inner_dict in outer_dict.items():
        if isinstance(inner_dict, dict):
            for inner_key, inner_value in inner_dict.items():
                if inner_value == value:
                    return inner_key
    return None

for chr in chr_dkey_ID_value:
    for dkey in chr_dkey_ID_value[chr]:
        ID_value = chr_dkey_ID_value[chr][dkey]
        ID_loc = {ID:ID[1:] for ID in ID_value}
        max_pos = genome_size[chr]

        binID_mean = statis.rbin_data_mean(bin_size=bin_size,
                                           bin_step=bin_step,
                                           ID_loc=ID_loc,
                                           ID_value=ID_value,
                                           max_pos=max_pos,
                                           silent=False)

        sig = [binID_mean[binID] for binID in sorted(binID_mean.keys())]
        sig = statis.slow_moving_average2(sig, blur_win)

        if chr not in chr_dkey_sig:
            chr_dkey_sig[chr] = {}
        chr_dkey_sig[chr][dkey] = sig

        prefix = _find_inner_key_by_value(dinfo_dkey, dkey)
        bg_fname = working_dir + f'Oct2025_run/analysis_v2/bigwig/{prefix}_binsize{bin_size}_smoothed.bdg'
        
        if output_bdg:
            # output bedGraph file
            if chr == 'chr1':
                file_handle = open(bg_fname, 'w')
            else:
                file_handle = open(bg_fname, 'a')
        
            for binID in sorted(binID_mean.keys()):
                value = binID_mean[binID]
                if np.isnan(value):
                    continue
                start_pos = binID * bin_size
                end_pos = start_pos + bin_size
                file_handle.write(f"{chr}\t{start_pos}\t{end_pos}\t{value}\n")
        # else:
        #     print(f"File already exists: {bg_fname}")
        #     continue

if output_bdg and output_bigwig:
    for inner_dict in dinfo_dkey.values():
        prefix = list(inner_dict.keys())[0]
        bg_fname = working_dir + f'Oct2025_run/analysis_v2/bigwig/{prefix}_binsize{bin_size}_smoothed.bdg'
        # convert bedGraph to bigWig
        bw_fname = bg_fname.replace('.bdg', '.bw')
        cmd = f"bedGraphToBigWig {bg_fname} {ref_path}/mm10.chrom.sizes {bw_fname}"
        print(f"Executing command: {cmd}")
        os.system(cmd)

file_handle.close()

hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash function is built
hash functi

Executing command: bedGraphToBigWig /Volumes/BackupHaLab/Oct2025_run/analysis_v2/bigwig/Exp1-Samp11_S2_binsize10000_smoothed.bdg /Volumes/BackupHaLab//genomes/mm10//mm10.chrom.sizes /Volumes/BackupHaLab/Oct2025_run/analysis_v2/bigwig/Exp1-Samp11_S2_binsize10000_smoothed.bw
Executing command: bedGraphToBigWig /Volumes/BackupHaLab/Oct2025_run/analysis_v2/bigwig/Exp1-Samp15_S3_binsize10000_smoothed.bdg /Volumes/BackupHaLab//genomes/mm10//mm10.chrom.sizes /Volumes/BackupHaLab/Oct2025_run/analysis_v2/bigwig/Exp1-Samp15_S3_binsize10000_smoothed.bw
Executing command: bedGraphToBigWig /Volumes/BackupHaLab/Oct2025_run/analysis_v2/bigwig/Exp2-Samp11_S5_binsize10000_smoothed.bdg /Volumes/BackupHaLab//genomes/mm10//mm10.chrom.sizes /Volumes/BackupHaLab/Oct2025_run/analysis_v2/bigwig/Exp2-Samp11_S5_binsize10000_smoothed.bw
Executing command: bedGraphToBigWig /Volumes/BackupHaLab/Oct2025_run/analysis_v2/bigwig/Exp2-Samp15_S6_binsize10000_smoothed.bdg /Volumes/BackupHaLab//genomes/mm10//mm10.chrom.si